# 05 Model Evaluation

Load saved models, compare predictions, plot residuals, and analyze errors by glucose range.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

from src.evaluate import regression_metrics
from src.predict import load_saved_models
from src.train_utils import load_feature_columns, time_series_train_test_split

PROCESSED_DIR = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"

In [ ]:
model_df = pd.read_csv(PROCESSED_DIR / "ohio_model_ready.csv", parse_dates=["timestamp"])
_, test_df = time_series_train_test_split(model_df, test_size=0.2)
feature_columns = load_feature_columns(MODELS_DIR / "feature_columns.json")
models = load_saved_models(MODELS_DIR)
target_columns = {
    "30min": "target_30min",
    "60min": "target_60min",
    "120min": "target_120min",
}

In [ ]:
prediction_frames = []
metric_rows = []

for horizon, target_column in target_columns.items():
    model = models[horizon]
    y_true = test_df[target_column]
    y_pred = model.predict(test_df[feature_columns])
    metric_rows.append({"horizon": horizon, **regression_metrics(y_true, y_pred)})

    frame = test_df[["patient_id", "timestamp", target_column]].copy()
    frame = frame.rename(columns={target_column: "actual_mgdl"})
    frame["predicted_mgdl"] = y_pred
    frame["residual_mgdl"] = frame["actual_mgdl"] - frame["predicted_mgdl"]
    frame["horizon"] = horizon
    prediction_frames.append(frame)

metrics_df = pd.DataFrame(metric_rows)
predictions_df = pd.concat(prediction_frames, ignore_index=True)
metrics_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=False, sharey=False)
for ax, horizon in zip(axes, target_columns):
    sample = predictions_df[predictions_df["horizon"] == horizon].sample(
        min(1000, (predictions_df["horizon"] == horizon).sum()), random_state=42
    )
    ax.scatter(sample["actual_mgdl"], sample["predicted_mgdl"], alpha=0.3)
    limits = [sample[["actual_mgdl", "predicted_mgdl"]].min().min(), sample[["actual_mgdl", "predicted_mgdl"]].max().max()]
    ax.plot(limits, limits, color="black", linestyle="--", linewidth=1)
    ax.set_title(f"Actual vs predicted {horizon}")
    ax.set_xlabel("Actual mg/dL")
    ax.set_ylabel("Predicted mg/dL")
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=False, sharey=True)
for ax, horizon in zip(axes, target_columns):
    residuals = predictions_df[predictions_df["horizon"] == horizon]["residual_mgdl"]
    ax.hist(residuals, bins=50)
    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_title(f"Residuals {horizon}")
    ax.set_xlabel("Actual - predicted mg/dL")
plt.tight_layout()

In [ ]:
def glucose_band(mgdl):
    mmol = mgdl / 18
    if mmol < 3.9:
        return "low"
    if mmol > 10:
        return "high"
    return "normal"

predictions_df["actual_band"] = predictions_df["actual_mgdl"].map(glucose_band)
band_errors = predictions_df.assign(abs_error=lambda df: df["residual_mgdl"].abs()).groupby(
    ["horizon", "actual_band"]
).agg(
    rows=("abs_error", "size"),
    mae_mgdl=("abs_error", "mean"),
    rmse_mgdl=("residual_mgdl", lambda s: np.sqrt(np.mean(s**2))),
).reset_index()
band_errors